# LieLines Predictions — Analysis

Analyses `sentence_corpus_predicted.csv` produced by `run_lielines.py`.
All heavy aggregations use chunked reads so the full 20 GB file is never loaded into memory.
Plots are produced from compact summary DataFrames.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PRED_FILE  = '/home/tom/data/sentence_corpus_predicted.csv'
CHUNK_SIZE = 500_000

print(f'File size: {os.path.getsize(PRED_FILE)/1e9:.2f} GB')

## 1. Overview stats

In [ ]:
total_rows  = 0
total_lies  = 0

for chunk in pd.read_csv(
    PRED_FILE,
    usecols=['lie_label'],
    chunksize=CHUNK_SIZE,
):
    total_rows += len(chunk)
    total_lies += (chunk['lie_label'] == 1).sum()

lie_rate = total_lies / total_rows
print(f'Total sentences : {total_rows:>15,}')
print(f'Flagged as lies : {total_lies:>15,}  ({lie_rate*100:.3f}%)')
print(f'Not flagged     : {total_rows - total_lies:>15,}')

## 2. Lie rate by country

In [ ]:
country_agg = {}   # country -> [total, lies]

for chunk in pd.read_csv(
    PRED_FILE,
    usecols=['country', 'lie_label'],
    chunksize=CHUNK_SIZE,
):
    chunk['lie_label'] = pd.to_numeric(chunk['lie_label'], errors='coerce').fillna(0).astype(int)
    grp = chunk.groupby('country')['lie_label'].agg(['count', 'sum'])
    for country, row in grp.iterrows():
        if country not in country_agg:
            country_agg[country] = [0, 0]
        country_agg[country][0] += row['count']
        country_agg[country][1] += row['sum']

df_country = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in country_agg.items()],
    columns=['country', 'total', 'lies']
).set_index('country')
df_country['lie_rate'] = df_country['lies'] / df_country['total']
df_country = df_country.sort_values('lie_rate', ascending=False)
print(df_country.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: lie rate per country
ax = axes[0]
df_country['lie_rate'].sort_values(ascending=True).plot(
    kind='barh', ax=ax, color=sns.color_palette('muted')[0]
)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=2))
ax.set_title('Lie accusation rate by country', fontsize=13)
ax.set_xlabel('% sentences flagged')
ax.set_ylabel('')

# Right: absolute counts (log scale)
ax2 = axes[1]
df_country[['lies', 'total']].sort_values('total', ascending=True).plot(
    kind='barh', ax=ax2, logx=True
)
ax2.set_title('Sentence counts by country (log scale)', fontsize=13)
ax2.set_xlabel('Count (log scale)')
ax2.set_ylabel('')
ax2.legend(['Flagged', 'Total'])

plt.tight_layout()
plt.savefig('lie_rate_by_country.png', bbox_inches='tight')
plt.show()

## 3. Lie rate over time (all countries combined)

In [ ]:
year_agg = {}   # year -> [total, lies]

for chunk in pd.read_csv(
    PRED_FILE,
    usecols=['date', 'lie_label'],
    chunksize=CHUNK_SIZE,
):
    chunk['lie_label'] = pd.to_numeric(chunk['lie_label'], errors='coerce').fillna(0).astype(int)
    chunk['year'] = pd.to_datetime(chunk['date'], errors='coerce').dt.year
    chunk = chunk.dropna(subset=['year'])
    chunk['year'] = chunk['year'].astype(int)
    grp = chunk.groupby('year')['lie_label'].agg(['count', 'sum'])
    for year, row in grp.iterrows():
        if year not in year_agg:
            year_agg[year] = [0, 0]
        year_agg[year][0] += row['count']
        year_agg[year][1] += row['sum']

df_time = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in year_agg.items()],
    columns=['year', 'total', 'lies']
).set_index('year').sort_index()
df_time['lie_rate'] = df_time['lies'] / df_time['total']
print(df_time)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_time.index, df_time['lie_rate'] * 100,
        marker='o', linewidth=2, markersize=4, color=sns.color_palette('muted')[1])
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f%%'))
ax.set_xlabel('Year')
ax.set_ylabel('% sentences flagged as lie accusation')
ax.set_title('Lie accusation rate over time (all countries)', fontsize=13)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('lie_rate_over_time.png', bbox_inches='tight')
plt.show()

## 4. Lie rate over time — per country

In [ ]:
cy_agg = {}   # (country, year) -> [total, lies]

for chunk in pd.read_csv(
    PRED_FILE,
    usecols=['date', 'country', 'lie_label'],
    chunksize=CHUNK_SIZE,
):
    chunk['lie_label'] = pd.to_numeric(chunk['lie_label'], errors='coerce').fillna(0).astype(int)
    chunk['year'] = pd.to_datetime(chunk['date'], errors='coerce').dt.year
    chunk = chunk.dropna(subset=['year'])
    chunk['year'] = chunk['year'].astype(int)
    grp = chunk.groupby(['country', 'year'])['lie_label'].agg(['count', 'sum'])
    for (country, year), row in grp.iterrows():
        key = (country, year)
        if key not in cy_agg:
            cy_agg[key] = [0, 0]
        cy_agg[key][0] += row['count']
        cy_agg[key][1] += row['sum']

df_cy = pd.DataFrame(
    [(k[0], k[1], v[0], v[1]) for k, v in cy_agg.items()],
    columns=['country', 'year', 'total', 'lies']
)
df_cy['lie_rate'] = df_cy['lies'] / df_cy['total']

# Only keep country-years with enough data to be meaningful
MIN_SENTENCES = 1000
df_cy = df_cy[df_cy['total'] >= MIN_SENTENCES]
print(f'Country-year combinations: {len(df_cy):,}')

In [ ]:
countries = sorted(df_cy['country'].unique())
ncols = 4
nrows = int(np.ceil(len(countries) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.2), sharey=False)
axes = axes.flatten()

palette = sns.color_palette('tab20', n_colors=len(countries))

for i, country in enumerate(countries):
    ax = axes[i]
    sub = df_cy[df_cy['country'] == country].sort_values('year')
    ax.plot(sub['year'], sub['lie_rate'] * 100,
            marker='o', linewidth=1.8, markersize=3, color=palette[i])
    ax.set_title(country, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f%%'))
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Lie accusation rate over time — by country', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('lie_rate_by_country_over_time.png', bbox_inches='tight')
plt.show()

## 5. Lie rate by source dataset

In [ ]:
ds_agg = {}

for chunk in pd.read_csv(
    PRED_FILE,
    usecols=['source_dataset', 'source_dataset_type', 'lie_label'],
    chunksize=CHUNK_SIZE,
):
    chunk['lie_label'] = pd.to_numeric(chunk['lie_label'], errors='coerce').fillna(0).astype(int)
    grp = chunk.groupby(['source_dataset', 'source_dataset_type'])['lie_label'].agg(['count', 'sum'])
    for (ds, dt), row in grp.iterrows():
        key = (ds, dt)
        if key not in ds_agg:
            ds_agg[key] = [0, 0]
        ds_agg[key][0] += row['count']
        ds_agg[key][1] += row['sum']

df_ds = pd.DataFrame(
    [(k[0], k[1], v[0], v[1]) for k, v in ds_agg.items()],
    columns=['source_dataset', 'source_dataset_type', 'total', 'lies']
)
df_ds['lie_rate'] = df_ds['lies'] / df_ds['total']
df_ds = df_ds.sort_values('lie_rate', ascending=False)
print(df_ds.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(5, len(df_ds) * 0.35)))
colors = df_ds['source_dataset_type'].map(
    {t: c for t, c in zip(df_ds['source_dataset_type'].unique(),
                          sns.color_palette('tab10', n_colors=df_ds['source_dataset_type'].nunique()))}
)
df_ds.sort_values('lie_rate', ascending=True).plot(
    kind='barh', x='source_dataset', y='lie_rate', ax=ax,
    color=colors.sort_values().values, legend=False
)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=3))
ax.set_title('Lie accusation rate by source dataset', fontsize=13)
ax.set_xlabel('% sentences flagged')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('lie_rate_by_dataset.png', bbox_inches='tight')
plt.show()

## 6. Score distribution

In [ ]:
# Sample 2M rows for the score histogram (no need to scan the whole file)
df_scores = pd.read_csv(PRED_FILE, usecols=['lie_label', 'lie_score'], nrows=2_000_000)
df_scores['lie_label'] = pd.to_numeric(df_scores['lie_label'], errors='coerce')
df_scores['lie_score'] = pd.to_numeric(df_scores['lie_score'], errors='coerce')
df_scores = df_scores.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: score histogram for both classes
for label, color, name in [(0, 'steelblue', 'Not lie'), (1, 'tomato', 'Lie')]:
    subset = df_scores[df_scores['lie_label'] == label]['lie_score']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=f'{name} (n={len(subset):,})', density=True)
axes[0].set_xlabel('Lie score (model confidence)')
axes[0].set_ylabel('Density')
axes[0].set_title('Score distribution by predicted class', fontsize=12)
axes[0].legend()

# Right: CDF of lie scores for flagged sentences
flagged_scores = df_scores[df_scores['lie_label'] == 1]['lie_score'].sort_values()
axes[1].plot(flagged_scores.values, np.linspace(0, 1, len(flagged_scores)), color='tomato')
axes[1].set_xlabel('Lie score threshold')
axes[1].set_ylabel('Fraction of flagged sentences below threshold')
axes[1].set_title('CDF of lie scores (flagged sentences only)', fontsize=12)
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('lie_score_distribution.png', bbox_inches='tight')
plt.show()

## 7. Top-scoring examples

In [ ]:
# Collect the top N highest-scoring lie sentences without loading the full file
import heapq

TOP_N = 50
top_heap = []  # min-heap of (score, row_dict)

for chunk in pd.read_csv(
    PRED_FILE,
    usecols=['sentence', 'date', 'speaker', 'country',
             'source_dataset', 'lie_label', 'lie_score'],
    chunksize=CHUNK_SIZE,
    dtype={'lie_label': 'Int64'},
):
    chunk['lie_score'] = pd.to_numeric(chunk['lie_score'], errors='coerce')
    flagged = chunk[chunk['lie_label'] == 1].dropna(subset=['lie_score'])
    for _, row in flagged.iterrows():
        score = row['lie_score']
        entry = (score, row.to_dict())
        if len(top_heap) < TOP_N:
            heapq.heappush(top_heap, entry)
        elif score > top_heap[0][0]:
            heapq.heapreplace(top_heap, entry)

top_df = pd.DataFrame([x[1] for x in sorted(top_heap, reverse=True)])
top_df[['lie_score', 'country', 'date', 'speaker', 'source_dataset', 'sentence']]

In [ ]:
# Print top 20 for readability
print('=== TOP 20 HIGHEST-SCORING LIE ACCUSATIONS ===\n')
for i, row in top_df.head(20).iterrows():
    print(f"[{row['lie_score']:.4f}] {row['country']} | {row['date']} | {row['speaker']}")
    print(f"  {row['sentence']}")
    print()

## 8. Heatmap — lie rate by country × decade

In [ ]:
df_cy['decade'] = (df_cy['year'] // 10 * 10).astype(str) + 's'

pivot = df_cy.groupby(['country', 'decade']).apply(
    lambda x: x['lies'].sum() / x['total'].sum()
).unstack('decade') * 100

# Sort countries by overall lie rate
overall_order = df_country['lie_rate'].sort_values(ascending=False).index
pivot = pivot.reindex([c for c in overall_order if c in pivot.index])

fig, ax = plt.subplots(figsize=(max(8, pivot.shape[1] * 1.5), max(5, pivot.shape[0] * 0.5)))
sns.heatmap(
    pivot, ax=ax, annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': '% sentences flagged'}
)
ax.set_title('Lie accusation rate (%) — country × decade', fontsize=13)
ax.set_xlabel('Decade')
ax.set_ylabel('Country')
plt.tight_layout()
plt.savefig('lie_rate_heatmap.png', bbox_inches='tight')
plt.show()